# 00. ChEMBL Bioactivity Prediction - Instalacja i Przegląd

## Projekt: Model AI przewidywania bioaktywności molekuł

### Co to za projekt?
Celem jest zbudowanie modelu uczenia maszynowego, który **przewiduje aktywność biologiczną cząsteczek chemicznych**. To może pomóc w odkrywaniu nowych leków bez konieczności kosztownego syntetyzowania i testowania tysięcy związków w laboratorium.

**Innymi słowy**: Celem jest nauczyć komputer rozpoznawać, które cząsteczki będą dobrymi kandydatami na leki.

---

### Struktura Projektu:
- **00_project_setup.ipynb** - Instalacja bibliotek, wprowadzenie, wiedza domenowa
- **01_data_acquisition.ipynb** - Pozyskiwanie i wstępna analiza danych ChEMBL
- **02_data_cleaning_eda.ipynb** - Czyszczenie danych i eksploracyjna analiza

---

### Dane:
- **Źródło**: ChEMBL 36 - największa publiczna baza danych bioaktywności
- **Rozmiar bazy**: ~2.9M molekuł, ~24.3M pomiarów bioaktywności
- **Cel**: Przewidywanie bioaktywności dla człowieka (Homo sapiens)
- **Fokus**: Pomiary IC50, Ki, Kd, EC50, Potency w nanomolach (nM)

---

### Stack Technologiczny:
- **Data Processing**: pandas, numpy, SQLAlchemy
- **Cheminformatics**: RDKit (obliczanie cech chemicznych z SMILES)
- **Visualization**: matplotlib, seaborn
- **Machine Learning**: scikit-learn, XGBoost, LightGBM
- **Tracking**: MLflow, pickle

---

### Zawartość tego notebooka:
1. **Wiedza domenowa** - wyjaśnienie pojęć farmakologicznych
2. **Setup środowiska** - instalacja bibliotek
3. **Struktura katalogów** - organizacja projektu
4. **Ekstrakcja bazy ChEMBL** - rozpakowanie danych

---
## Wiedza Domenowa

### Czym jest odkrywanie leków (Drug Discovery)?
Odkrywanie nowych leków to proces poszukiwania cząsteczek chemicznych, które mogą leczyć choroby. Jest to jak szukanie "klucza" (molekuły leku), który pasuje do "zamka" (białka w organizmie związanego z chorobą). Ten proces może trwać 10-15 lat i kosztować nawet miliardy dolarów.

### Kluczowe Pojęcia:

#### 1. **Target (Cel Biologiczny)**

- To białko lub molekuła w ciele, która jest związana z chorobą
- Przykład: enzym, receptor, kanał jonowy
- **Analogia**: Target to "zamek", który chcemy otworzyć lub zablokować

#### 2. **Bioaktywność (Bioactivity)**

- To miara tego, jak silnie cząsteczka chemiczna oddziałuje z targetem
- **Im niższa wartość (w nM), tym silniejsze oddziaływanie!**
- Analogia: To jak mocno klucz pasuje do zamka

#### 3. **Jednostki Pomiarów:**

**nM (nanomol/litr)** = nanomolar
- To koncentracja cząsteczki potrzebna do wywołania efektu
- 1 nM = 0.000000001 mol/litr
- **Zasada**: Niższa wartość = lepsza cząsteczka (bardziej aktywna)
- Przykłady:
  - 1 nM = doskonała aktywność (potencjalny kandydat na lek)
  - 100 nM = dobra aktywność
  - 10,000 nM (10 μM) = słaba aktywność

#### 4. **Typy Pomiarów Bioaktywności:**

**IC50** (Half Maximal Inhibitory Concentration)
- Koncentracja, przy której cząsteczka blokuje 50% aktywności targetu
- Najczęściej używany pomiar w testach screeningowych
- **Analogia**: Ile trucizny potrzeba, żeby zahamować połowę bakterii

**Ki** (Inhibition Constant)
- Mierzy jak mocno inhibitor wiąże się z targetem
- Bardziej fundamentalna wartość niż IC50
- **Analogia**: Jak mocno klucz "przykleja się" do zamka

**Kd** (Dissociation Constant)
- Mierzy siłę wiązania między cząsteczką a targetem
- Im niższe Kd, tym silniejsze wiązanie
- **Analogia**: Jak trudno jest oderwać magnes od lodówki

**EC50** (Half Maximal Effective Concentration)
- Koncentracja dająca 50% maksymalnego efektu
- Używana dla aktywatorów (agoniści), nie inhibitorów
- **Analogia**: Ile benzyny potrzebujesz, żeby samochód jechał z połową maksymalnej prędkości

#### 5. **SMILES (Simplified Molecular Input Line Entry System)**

- To tekstowy zapis struktury chemicznej molekuły
- Pozwala komputerowi "zrozumieć" jak wygląda cząsteczka
- Przykład: `CC(=O)O` to kwas octowy (ocet)
- **Analogia**: To jak kod kreskowy dla molekuł

#### 6. **ChEMBL Database**
- Największa publiczna baza danych bioaktywności
- Zawiera miliony pomiarów dla milionów cząsteczek
- Dane z publikacji naukowych i testów klinicznych
- **Analogia**: To jak "Wikipedia" dla danych farmakologicznych

#### 7. **Assay (Test Biochemiczny)**
- Eksperyment laboratoryjny mierzący aktywność cząsteczki
- Może być:
  - **Binding assay** = czy cząsteczka wiąże się z targetem?
  - **Functional assay** = czy cząsteczka zmienia działanie targetu?
- **Assay Confidence Score (0-9)**: Jak pewni jesteśmy, że test zmierzył to, co chcieliśmy
  - 9 = target potwierdzony krystalografią (super pewne)
  - 7 = pojedyncze białko, dobrze scharakteryzowane
  - 4 = mniej pewny target lub kompleks białek

### Dlaczego Uczenie Maszynowe w farmakologii?

1. **Przewidywanie zamiast testowania**: Chcemy nauczyć komputer przewidywać, które nowe cząsteczki będą aktywne, zanim je zsyntetyzujemy w laboratorium

2. **Oszczędność czasu i pieniędzy**: Synteza i testowanie jednej cząsteczki może kosztować nawet tysiące złotych. Uczenie Maszynowe pozwala "testować" miliony cząsteczek wirtualnie

3. **Przyspieszenie odkryć**: Zamiast testować 1000 losowych cząsteczek, można przetestować 10 najlepszych kandydatów wybranych przez model

### Pipeline projektu:

1. **01: Zbiór danych** o istniejących cząsteczkach i ich aktywności
2. **02: Czyszczenie dane** - usuwanie błędów i niepełnych rekordów
3. **03: Tworzymy cechy** - tzn. Feature Engineering
4. **04: Budujemy model ML** - nauka komputera (modelu) przewidywać aktywność nowych cząsteczek

### Przykład praktyczny:

Wyobraźmy sobie, że szukamy leku na Alzheimera:
1. **Target**: Enzym beta-sekretaza (powoduje chorobę)
2. **Cel**: Znaleźć cząsteczkę, która hamuje ten enzym
3. **Dane**: Mamy 100,000 cząsteczek z pomiarami IC50
4. **Model**: Uczymy komputer, które właściwości cząsteczki prowadzą do niskiego IC50
5. **Predykcja**: Model proponuje nowe cząsteczki do syntezy
6. **Walidacja**: Syntetyzujemy top 10 kandydatów i testujemy w laboratorium

## Instalacja Zależności

Jeśli poniższe biblioteki nie są zainstalowane, należy odkomentować poniższy kod, a następnia go uruchomić:

In [ ]:
# Instalacja podstawowych bibliotek

# %pip install pandas numpy matplotlib seaborn
# %pip install sqlalchemy
# %pip install rdkit
# %pip install scikit-learn xgboost lightgbm
# %pip install chembl_webresource_client
# %pip install dask[complete]
# %pip install mlflow
# %pip install tqdm

## Ekstrakcja Bazy Danych SQLite

Jeśli baza danych nie jest jeszcze rozpakowana należy odkomentować poniższy kod i go uruchomić:

**Upewnij się, że zapakowana baza danych znajduje się w głównym katalogu projektu**

In [ ]:
import tarfile
import shutil
from pathlib import Path

base_path = r"<GŁÓWNY_KATALOG_PROJEKTU>"  # <-- Zmień na rzeczywistą ścieżkę do katalogu projektu
base_dir = Path(base_path)

tar_filename = "<NAZWA_PLIKU>.tar.gz" # <-- Zmień na rzeczywistą nazwę pliku tar.gz, jeśli jest inna
tar_file = base_dir / tar_filename

db_dirname = "<NAZWA_FOLDERU_DB>"  # <-- Zmień na nazwę folderu docelowego dla bazy danych, zalecane: "db"
db_dir = base_dir / db_dirname
tar_file = base_dir / tar_filename

target_db_name = "<NAZWA_PLIKU_DB>.db"  # <-- Zmień na rzeczywistą nazwę pliku bazy danych, np. "chembl_36.db"
target_db_path = db_dir / target_db_name

db_dir.mkdir(parents=True, exist_ok=True)

# Sprawdzenie, czy baza już istnieje w docelowej lokalizacji
if target_db_path.exists():
    print(f"Baza danych już istnieje w miejscu docelowym: {target_db_path}")
    print(f"  Rozmiar: {target_db_path.stat().st_size / (1024**3):.2f} GB")
else:
    print(f"Rozpakowywanie bazy danych bezpośrednio do folderu {db_dirname}/...")
    
    try:
        with tarfile.open(tar_file, "r:gz") as tar:
            db_member = None
            for member in tar.getmembers():
                if member.name.endswith('.db') and 'chembl' in member.name.lower():
                    db_member = member
                    print(f"  Znaleziono w archiwum: {member.name}")
                    break
            
            if db_member:
                print(f"  Rozpakowywanie do {target_db_path}...")
                
                with tar.extractfile(db_member) as source:
                    with open(target_db_path, 'wb') as target:
                        shutil.copyfileobj(source, target)
                
                print(f"Baza danych rozpakowana pomyślnie!")
                print(f"  Lokalizacja: {target_db_path}")
                print(f"  Rozmiar: {target_db_path.stat().st_size / (1024**3):.2f} GB")
            else:
                print("Uwaga: Nie znaleziono pliku bazy danych w archiwum!")
                print("   Sprawdź zawartość archiwum ręcznie.")
    
    except Exception as e:
        print(f"Uwaga: Błąd podczas rozpakowywania: {e}")
        print("   Sprawdź czy plik tar.gz istnieje i jest poprawny.")

print(f"\nŚcieżka do bazy danych: {target_db_path}")

Rozpakowywanie bazy danych bezpośrednio do folderu db/...
  Znaleziono w archiwum: chembl_36/chembl_36_sqlite/chembl_36.db
  Rozpakowywanie do /Users/lukasz/Documents/Python/chembl/db/chembl_36.db...
Baza danych rozpakowana pomyślnie!
  Lokalizacja: /Users/lukasz/Documents/Python/chembl/db/chembl_36.db
  Rozmiar: 27.70 GB

Ścieżka do bazy danych: /Users/lukasz/Documents/Python/chembl/db/chembl_36.db


## Struktura Katalogów Projektu

Stwórzmy organizację katalogów:

In [3]:
# Struktura katalogów
directories = [
    "db",                 # Baza danych ChEMBL SQLite
    "data/raw",           # Surowe dane z ChEMBL
    "data/processed",     # Wyczyszczone dane
    "notebooks",          # Notebooki (opcjonalnie)
]

for directory in directories:
    dir_path = base_dir / directory
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"- Utworzono: {directory}")

print("\nStruktura katalogów utworzona pomyślnie.")

- Utworzono: db
- Utworzono: data/raw
- Utworzono: data/processed
- Utworzono: notebooks

Struktura katalogów utworzona pomyślnie.


> Następny notebook: [01_data_exploration.ipynb](01_data_exploration.ipynb)